In [29]:
# Check if required packages are available
import sys
print(f"Python version: {sys.version}")
print(f"Python executable: {sys.executable}")

try:
    import transformers
    print(f"✓ transformers version: {transformers.__version__}")
except ImportError:
    print("✗ transformers not found")

try:
    import datasets
    print(f"✓ datasets version: {datasets.__version__}")
except ImportError:
    print("✗ datasets not found")

try:
    import evaluate
    print("✓ evaluate module found")
except ImportError:
    print("✗ evaluate not found")

try:
    import rouge_score
    print("✓ rouge_score module found")
except ImportError:
    print("✗ rouge_score not found")

try:
    import accelerate
    print(f"✓ accelerate version: {accelerate.__version__}")
except ImportError:
    print("✗ accelerate not found")

print("\nAll required packages should be installed in the virtual environment.")


Python version: 3.13.5 (main, Jun 21 2025, 09:35:00) [GCC 15.1.1 20250425]
Python executable: /home/ray/Documents/Tugas Kuliah/Semester 8/NLP/Proyek Semester/newsum-model/.venv/bin/python
✓ transformers version: 4.53.1
✓ datasets version: 3.6.0
✓ evaluate module found
✓ rouge_score module found
✓ accelerate version: 1.8.1

All required packages should be installed in the virtual environment.


In [48]:
import os
import json
import torch
from tqdm import tqdm
from transformers import BertTokenizer, EncoderDecoderModel, Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq
from datasets import Dataset
import evaluate

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

# Set device to CPU
device = torch.device('cpu')
print(f"Using device: {device}")

# Disable accelerate by setting environment variable
os.environ['ACCELERATE_DISABLE_RICH'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'


PyTorch version: 2.7.1+cu126
CUDA available: False
Device: cpu
Using device: cpu


In [53]:
tokenizer = BertTokenizer.from_pretrained("cahya/bert2bert-indonesian-summarization")
tokenizer.bos_token = tokenizer.cls_token
tokenizer.eos_token = tokenizer.sep_token

model = EncoderDecoderModel.from_pretrained("cahya/bert2bert-indonesian-summarization")


In [54]:
import tarfile
import os

# Use the local dataset file from the root folder
tar_path = "liputan6_data.tar.gz"
extract_path = "./dataset"

# Create the dataset directory if it doesn't exist
os.makedirs(extract_path, exist_ok=True)

# Extract the file to ./dataset
with tarfile.open(tar_path, "r:gz") as tar:
    tar.extractall(path=extract_path)

/tmp/ipykernel_180843/1374363040.py:13: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=extract_path)


In [55]:
import os
import json
from tqdm import tqdm

def load_dataset_from_folder(folder_path, limit=None):
    data = []
    files = sorted(os.listdir(folder_path))[:limit]
    for file_name in tqdm(files):
        with open(os.path.join(folder_path, file_name), 'r', encoding='utf-8') as f:
            item = json.load(f)
            input_text = " ".join([" ".join(sent) for sent in item["clean_article"]])
            summary_text = " ".join([" ".join(sent) for sent in item["clean_summary"]])
            data.append({
                "article": input_text,
                "summary": summary_text
            })
    return data

# Load 1600 data train dan 400 data test using local dataset
train_data = load_dataset_from_folder("./dataset/liputan6_data/xtreme/dev", limit=1600)
test_data = load_dataset_from_folder("./dataset/liputan6_data/xtreme/dev", limit=400)

  0%|          | 0/1600 [00:00<?, ?it/s]

100%|██████████| 400/400 [00:00<00:00, 5069.87it/s]


In [56]:
from datasets import Dataset

train_dataset = Dataset.from_list(train_data)
test_dataset = Dataset.from_list(test_data)

In [57]:
max_input_length = 512
max_target_length = 128

def preprocess_function(examples):
    inputs = tokenizer(examples["article"], max_length=max_input_length, padding="max_length", truncation=True)
    targets = tokenizer(examples["summary"], max_length=max_target_length, padding="max_length", truncation=True)

    inputs["labels"] = targets["input_ids"]
    return inputs

tokenized_train = train_dataset.map(preprocess_function, batched=True)
tokenized_test = test_dataset.map(preprocess_function, batched=True)

Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

In [58]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

In [26]:
rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    return {key: value * 100 for key, value in result.items()}

In [50]:
import os
import torch

# Manual training configuration
class SimpleTrainingConfig:
    def __init__(self):
        self.output_dir = "./results"
        self.learning_rate = 2e-5
        self.batch_size = 1  # Very small for CPU
        self.num_epochs = 1
        self.device = torch.device('cpu')
        
config = SimpleTrainingConfig()
print(f"Training configuration created for device: {config.device}")
print(f"Batch size: {config.batch_size}")
print(f"Learning rate: {config.learning_rate}")

# Create output directory
os.makedirs(config.output_dir, exist_ok=True)


Training configuration created for device: cpu
Batch size: 1
Learning rate: 2e-05


In [59]:
# Manual training setup
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.nn.utils import clip_grad_norm_

# Move model to CPU
model = model.to(config.device)
print(f"Model moved to {config.device}")

# Remove text columns to avoid collator issues
tokenized_train_clean = tokenized_train.remove_columns(['article', 'summary'])
tokenized_test_clean = tokenized_test.remove_columns(['article', 'summary'])

print("Removed text columns from tokenized datasets")

# Create data loaders with cleaned datasets
train_loader = DataLoader(
    tokenized_train_clean, 
    batch_size=config.batch_size, 
    shuffle=True,
    collate_fn=data_collator
)

eval_loader = DataLoader(
    tokenized_test_clean, 
    batch_size=config.batch_size, 
    shuffle=False,
    collate_fn=data_collator
)

# Setup optimizer
optimizer = AdamW(model.parameters(), lr=config.learning_rate)

print(f"Training data loader created: {len(train_loader)} batches")
print(f"Evaluation data loader created: {len(eval_loader)} batches")
print("Optimizer configured")

# Prepare for training
model.train()
total_steps = len(train_loader) * config.num_epochs
print(f"Total training steps: {total_steps}")
print("Starting training...")

for epoch in range(config.num_epochs):
    for step, batch in enumerate(train_loader):
        # Zero the gradients
        optimizer.zero_grad()

        # Forward pass
        inputs = {k: v.to(config.device) for k, v in batch.items()}
        outputs = model(**inputs)

        # Compute loss
        loss = outputs.loss
        print(f"Epoch {epoch}, Step {step}, Loss: {loss.item()}")

        # Backward pass
        loss.backward()

        # Clip gradients
        clip_grad_norm_(model.parameters(), max_norm=1.0)

        # Update parameters
        optimizer.step()


Model moved to cpu
Removed text columns from tokenized datasets
Training data loader created: 1600 batches
Evaluation data loader created: 400 batches
Optimizer configured
Total training steps: 1600
Starting training...


/home/ray/Documents/Tugas Kuliah/Semester 8/NLP/Proyek Semester/newsum-model/.venv/lib/python3.13/site-packages/transformers/models/encoder_decoder/modeling_encoder_decoder.py:577: FutureWarning: Version v4.12.0 introduces a better way to train encoder-decoder models by computing the loss inside the encoder-decoder framework rather than in the decoder itself. You may observe training discrepancies if fine-tuning a model trained with versions anterior to 4.12.0. The decoder_input_ids are now created based on the labels, no need to pass them yourself anymore.
  warnings.warn(DEPRECATION_WARNING, FutureWarning)


Epoch 0, Step 0, Loss: 18.114730834960938
Epoch 0, Step 1, Loss: 15.451784133911133
Epoch 0, Step 1, Loss: 15.451784133911133
Epoch 0, Step 2, Loss: 9.395811080932617
Epoch 0, Step 2, Loss: 9.395811080932617
Epoch 0, Step 3, Loss: 6.29504919052124
Epoch 0, Step 3, Loss: 6.29504919052124
Epoch 0, Step 4, Loss: 3.1203718185424805
Epoch 0, Step 4, Loss: 3.1203718185424805
Epoch 0, Step 5, Loss: 2.6660101413726807
Epoch 0, Step 5, Loss: 2.6660101413726807
Epoch 0, Step 6, Loss: 1.1949337720870972
Epoch 0, Step 6, Loss: 1.1949337720870972
Epoch 0, Step 7, Loss: 0.6984924674034119
Epoch 0, Step 7, Loss: 0.6984924674034119
Epoch 0, Step 8, Loss: 0.7595831155776978
Epoch 0, Step 8, Loss: 0.7595831155776978
Epoch 0, Step 9, Loss: 0.9126729965209961
Epoch 0, Step 9, Loss: 0.9126729965209961
Epoch 0, Step 10, Loss: 0.7826452255249023
Epoch 0, Step 10, Loss: 0.7826452255249023
Epoch 0, Step 11, Loss: 0.9903018474578857
Epoch 0, Step 11, Loss: 0.9903018474578857
Epoch 0, Step 12, Loss: 0.9480026364

KeyboardInterrupt: 

In [ ]:
trainer.train()


In [ ]:
results = trainer.evaluate()
print(results)

In [ ]:
model.save_pretrained("./indo_summary_model")
tokenizer.save_pretrained("./indo_summary_model")
